In [1]:
# verify the number of constraint lower and upper bound

In [2]:
from robot import Robot, MotomanRobot
from planning_scene import PlanningScene
from geometric_trajopt_ipopt import PoseTrajOpt
import numpy as np
import scipy as sp
import transformations as tf


In [3]:
# test the robot with the scene
# add environment collisions
pcd_total = []
# shelf-bottom
num_points = 1000
position = np.array([0.85, 0, 0.5])
half_size = np.array([0.175, 0.5, 0.5])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-top
num_points = 1000
position = np.array([0.85, 0, 1.42])
half_size = np.array([0.175, 0.5, 0.025])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-left
num_points = 1000
position = np.array([0.85, -0.475, 1.2])
half_size = np.array([0.175, 0.025, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-right
num_points = 1000
position = np.array([0.85, 0.475, 1.2])
half_size = np.array([0.175, 0.025, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-back
num_points = 1000
position = np.array([1.0, 0, 1.2])
half_size = np.array([0.025, 0.5, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
pcd_total = np.concatenate(pcd_total, axis=0)


torso_b1 = ["torso_joint_b1"]
left = [
    "arm_left_joint_1_s",
    "arm_left_joint_2_l",
    "arm_left_joint_3_e",
    "arm_left_joint_4_u",
    "arm_left_joint_5_r",
    "arm_left_joint_6_b",
    "arm_left_joint_7_t",
]
right = [
    "arm_right_joint_1_s",
    "arm_right_joint_2_l",
    "arm_right_joint_3_e",
    "arm_right_joint_4_u",
    "arm_right_joint_5_r",
    "arm_right_joint_6_b",
    "arm_right_joint_7_t",
]
robot_joint_names = right#torso_b1 + left + right
robot = MotomanRobot(selected_joint_names=robot_joint_names)
# scene = PlanningScene(robot, scene_pcd=pcd_total)
scene = PlanningScene(robot, scene_pcd=None)
scene.update_scene_pcd(pcd_total)


In [4]:
pos = np.array([0.9096643361780983, -0.22, 1.2246445275392399])
quat = np.array([0.99997882565292, 0.0020695812292800746, -0.005795312726922581, 0.0021164663332217232])
pose = tf.quaternion_matrix(quat)
pose[:3,3] = pos
n_waypoints = 3
start_q = np.zeros((len(robot.selected_joint_dofids)))
solver = PoseTrajOpt(robot, scene, start_q=start_q, target_link="arm_right_link_7_t", target_pose=pose, n_waypoints=n_waypoints)


lb shape:  (21,)
lb: 
[-3.14159 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159
 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159 -1.91986
 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159]
ub: 
[3.14159 1.91986 2.96706 2.35619 3.14159 1.91986 3.14159 3.14159 1.91986
 2.96706 2.35619 3.14159 1.91986 3.14159 3.14159 1.91986 2.96706 2.35619
 3.14159 1.91986 3.14159]
cl shape:  (1023,)
cl: 
[0.01 0.01 0.01 ... -inf -inf -inf]
cu: 
[       inf        inf        inf ... 0.08726646 0.08726646 0.08726646]


In [5]:
qs = np.random.uniform(robot.selected_joint_limits[:,0], robot.selected_joint_limits[:,1], size=(2, len(robot_joint_names)))
# qs = np.linspace(start=qs[0,:], stop=qs[1,:], num=n_waypoints+1)
# solver.set_start_q(qs[0,:])
# qs = qs[1:,:]
# print(solver.start_q)
# print(qs)

step = (qs[1]-qs[0]) / (n_waypoints+1)
q0 = qs[0,:]
qs = []
solver.set_start_q(q0)
qs.append(q0+step)
for i in range(n_waypoints-1):
    qs.append(qs[-1]+step)
qs = np.array(qs)

In [6]:
solver.compute_collision_constraints(qs.flatten())
solver_jacs = solver.prev_collision_jacs
solver_jacs.shape

[1.09984391e-04 9.99999966e-01 2.36212650e-04]
[ 1.36824512e-04 -9.99999968e-01  2.12917032e-04]
[ 0.08174181 -0.98282976  0.16542047]
[ 0.0817418  -0.98282977  0.16542043]
[ 0.08174058 -0.98282985  0.16542058]
[-0.991596    0.06704732  0.11064367]
[-0.72287683 -0.66414395  0.19068798]
[-0.41513901 -0.887727    0.19899845]
[-0.56225384 -0.80269199  0.19888737]
[ 0.97056412  0.18583135 -0.15320575]
[ 0.82508603 -0.5647762   0.01615204]
[ 0.55760995 -0.82528374  0.08931905]
[ 0.68909265 -0.72236486  0.05779554]
[1.09984391e-04 9.99999966e-01 2.36212650e-04]
[ 1.36824512e-04 -9.99999968e-01  2.12917032e-04]
[0.57361791 0.02337855 0.81878931]
[0.57361796 0.02337834 0.81878928]
[0.57361692 0.02337828 0.81879001]
[-0.87230723 -0.16901868  0.45881673]
[-0.26712124 -0.13268094  0.95448521]
[ 0.10666873 -0.07218868  0.9916706 ]
[-0.0607994  -0.10093419  0.9930336 ]
[ 0.72471733  0.16300348 -0.66948835]
[0.9843672  0.16191324 0.06932041]
[0.89311143 0.11293604 0.43542786]
[0.9508823  0.13721068 

(7140,)

In [7]:
def naive_finite_diff(func, x, eps=1e-8):
    # given a function and the point for differentiation, compute the naive finite diff
    # x is a numpy array of shape (n,)
    diff = eps
    grad = np.zeros(x.shape)
    for i in range(len(x)):
        x_plus = x.copy()
        x_plus[i] += diff
        x_minus = x.copy()
        x_minus[i] -= diff
        grad[i] = (func(x_plus) - func(x_minus)) / (2*diff)
    return grad

import time

def naive_finite_diff_jacobian(func, x, eps=1e-8):
    # given a function and the point for differentiation, compute the naive finite diff
    # func returns array of shape (M,)  #(M can be a list)
    # x is a numpy array of shape (n,)
    # result of shape (M,n)
    func_shape = list(func(x).shape)
    x_shape = list(x.shape)
    jacobian = np.zeros(list(func(x).shape)+list(x.shape)).reshape((np.prod(func_shape), np.prod(x_shape)))  # first flatten the func shape
    for i in range(np.prod(x_shape)):
        start_time = time.time()
        print('i: %d/%d' % (i, np.prod(x_shape)))
        x_plus = x.copy()
        x_plus.flat[i] += eps
        x_minus = x.copy()
        x_minus.flat[i] -= eps
        
        deriv = (func(x_plus) - func(x_minus)) / (2*eps)
        jacobian[:,i] = deriv.flatten()
        duration = time.time() - start_time
        print('time: %f' % (duration))
        print('time prediction needed: ', duration * (np.prod(x_shape)-1-i))
    jacobian = jacobian.reshape(func_shape+x_shape)
    return jacobian

In [8]:
# debug total gradient
grad = solver.gradient(qs.flatten())
grad_num = naive_finite_diff(solver.objective, qs.flatten())
print(np.linalg.norm(grad-grad_num))

waypoint distance:  3.84437768937667
terminal pose distance:  3.1656566064952285
waypoint distance:  3.84437768937667
terminal pose distance:  3.1656566064952285
waypoint distance:  3.8443776893766706
terminal pose distance:  3.1656566064952285
waypoint distance:  3.84437768937667
terminal pose distance:  3.1656566064952285
waypoint distance:  3.84437768937667
terminal pose distance:  3.1656566064952285
waypoint distance:  3.84437768937667
terminal pose distance:  3.1656566064952285
waypoint distance:  3.84437768937667
terminal pose distance:  3.1656566064952285
waypoint distance:  3.84437768937667
terminal pose distance:  3.1656566064952285
waypoint distance:  3.8443776893766706
terminal pose distance:  3.1656566064952285
waypoint distance:  3.84437768937667
terminal pose distance:  3.1656566064952285
waypoint distance:  3.84437768937667
terminal pose distance:  3.1656566064952285
waypoint distance:  3.8443776893766706
terminal pose distance:  3.1656566064952285
waypoint distance:  3.

In [9]:
# debug total jacobian
jac = solver.jacobian(qs.flatten())  # sparse structure
row, col = solver.jacobianstructure()
# jacs = sp.sparse.coo_array((jacs_data, (row, col)), shape=(self.n_waypoints*len(collision_results),
#                                                            self.n_waypoints*self.ndof))
jac_mat = sp.sparse.coo_array((jac, (row, col)), shape=(solver.m, solver.n_waypoints*solver.ndof))
jac_num = naive_finite_diff_jacobian(solver.constraints, qs.flatten())
print(np.linalg.norm(jac_mat-jac_num))

i: 0/21
time: 0.000291
time prediction needed:  0.005812644958496094
i: 1/21
time: 0.000165
time prediction needed:  0.003143787384033203
i: 2/21
time: 0.000230
time prediction needed:  0.004145622253417969
i: 3/21
time: 0.000156
time prediction needed:  0.002658843994140625
i: 4/21
time: 0.000155
time prediction needed:  0.002483367919921875
i: 5/21
time: 0.000152
time prediction needed:  0.0022852420806884766
i: 6/21
time: 0.000212
time prediction needed:  0.0029706954956054688
i: 7/21
time: 0.000154
time prediction needed:  0.002005338668823242
i: 8/21
time: 0.000150
time prediction needed:  0.0017995834350585938
i: 9/21
time: 0.000147
time prediction needed:  0.0016129016876220703
i: 10/21
time: 0.000147
time prediction needed:  0.001468658447265625
i: 11/21
time: 0.000151
time prediction needed:  0.0013604164123535156
i: 12/21
time: 0.000185
time prediction needed:  0.001483917236328125
i: 13/21
time: 0.000206
time prediction needed:  0.00144195556640625
i: 14/21
time: 0.000211
ti

In [10]:
collision_jac = solver.collision_jacobian(qs.flatten())
row, col = solver.collision_jacobian_structure()
collision_jac = sp.sparse.coo_array((collision_jac, (row, col)), shape=(solver.n_waypoints*solver.scene.col_pair_num, solver.n_waypoints*solver.ndof))

position_jac = solver.position_distance_jacobian(qs.flatten())
row, col = solver.position_distance_jacobian_structure()
position_jac = sp.sparse.coo_array((position_jac, (row, col)), shape=(solver.n_waypoints, solver.n_waypoints*solver.ndof))

# joint_limit_jac = solver.joint_limit_jacobian(qs.flatten())
# row, col = solver.joint_limit_jacobian_structure()
# joint_limit_jac = sp.sparse.coo_array((joint_limit_jac, (row, col)), shape=(solver.n_waypoints*solver.ndof, solver.n_waypoints*solver.ndof))


In [11]:
def collision_constrs(qs):
    solver.compute_collision_constraints(qs.flatten())
    return solver.prev_collision_constrs
collision_jac_num = naive_finite_diff_jacobian(collision_constrs, qs.flatten(), eps=1e-8)
position_jac_num = naive_finite_diff_jacobian(solver.position_distance_constraints, qs.flatten())
# joint_limit_jac_num = naive_finite_diff_jacobian(solver.joint_limit_constraints, qs.flatten())

[1.09984391e-04 9.99999966e-01 2.36212650e-04]
[ 1.36824512e-04 -9.99999968e-01  2.12917032e-04]
[ 0.08174181 -0.98282976  0.16542047]
[ 0.0817418  -0.98282977  0.16542043]
[ 0.08174058 -0.98282985  0.16542058]
[-0.991596    0.06704732  0.11064367]
[-0.72287683 -0.66414395  0.19068798]
[-0.41513901 -0.887727    0.19899845]
[-0.56225384 -0.80269199  0.19888737]
[ 0.97056412  0.18583135 -0.15320575]
[ 0.82508603 -0.5647762   0.01615204]
[ 0.55760995 -0.82528374  0.08931905]
[ 0.68909265 -0.72236486  0.05779554]
[1.09984391e-04 9.99999966e-01 2.36212650e-04]
[ 1.36824512e-04 -9.99999968e-01  2.12917032e-04]
[0.57361791 0.02337855 0.81878931]
[0.57361796 0.02337834 0.81878928]
[0.57361692 0.02337828 0.81879001]
[-0.87230723 -0.16901868  0.45881673]
[-0.26712124 -0.13268094  0.95448521]
[ 0.10666873 -0.07218868  0.9916706 ]
[-0.0607994  -0.10093419  0.9930336 ]
[ 0.72471733  0.16300348 -0.66948835]
[0.9843672  0.16191324 0.06932041]
[0.89311143 0.11293604 0.43542786]
[0.9508823  0.13721068 

In [12]:
collision_jac.toarray()

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.        ,  0.        ,  0.        , ..., -0.11173062,
         0.26314549, -0.03270688],
       [ 0.        ,  0.        ,  0.        , ...,  0.1826107 ,
         0.18683768, -0.0414453 ],
       [ 0.        ,  0.        ,  0.        , ...,  0.16900207,
         0.14627258, -0.0360995 ]])

In [14]:
print(np.linalg.norm(collision_jac-collision_jac_num))
print(np.linalg.norm(position_jac-position_jac_num))
# print(np.linalg.norm(joint_limit_jac-joint_limit_jac_num))
# this verifies that the collision jacobian is not exactly correct.

3.213449395264471
3.2413324457765084e-08


In [ ]:
# compare the time needed to compute the finite diff for collision constraints jacobian
delattr(solver, "prev_qs")
import time

start_time = time.time()
collision_jac = solver.collision_jacobian(qs.flatten())
print("Time for collision jacobian: ", time.time()-start_time)

start_time = time.time()
collision_jac_num = naive_finite_diff_jacobian(solver.collision_constraints, qs.flatten())
print("Time for collision jacobian: ", time.time()-start_time)
print(np.linalg.norm(collision_jac-collision_jac_num))

row: 
[   0.    0.    0. ... 1019. 1019. 1019.]
col:
[ 0  1  2 ... 18 19 20]
Time for collision jacobian:  0.05823087692260742
i: 0/21
time: 0.000144
time prediction needed:  0.0028753280639648438
i: 1/21
time: 0.000120
time prediction needed:  0.002278566360473633
i: 2/21
time: 0.000118
time prediction needed:  0.002124309539794922
i: 3/21
time: 0.000118
time prediction needed:  0.0019981861114501953
i: 4/21
time: 0.000117
time prediction needed:  0.00186920166015625
i: 5/21
time: 0.000137
time prediction needed:  0.002052783966064453
i: 6/21
time: 0.000135
time prediction needed:  0.0018858909606933594
i: 7/21
time: 0.000112
time prediction needed:  0.0014598369598388672
i: 8/21
time: 0.000116
time prediction needed:  0.0013933181762695312
i: 9/21
time: 0.000113
time prediction needed:  0.001245737075805664
i: 10/21
time: 0.000112
time prediction needed:  0.0011205673217773438
i: 11/21
time: 0.000112
time prediction needed:  0.0010106563568115234
i: 12/21
time: 0.000113
time predicti

In [ ]:
rtol = 1e-5
atol = 1e-8
x_plus = qs.flatten().copy()
i = 0
eps = 1e-3
x_plus.flat[i] += eps
x_minus = qs.flatten().copy()
x_minus.flat[i] -= eps
# check if they are close
np.allclose(x_plus, qs.flatten())


False

In [ ]:
row, col = solver.collision_jacobian_structure()
print(row)
print(col)

[   0    0    0 ... 1019 1019 1019]
[ 0  1  2 ... 18 19 20]


In [ ]:
# check the total jacobian